# 02 - Data Preprocessing and Feature Engineering

## ES | Objetivo

El objetivo de este notebook es transformar el dataset de burnout en datos listos para el entrenamiento de un modelo de Machine Learning.

A partir de los hallazgos del notebook `01_business_oriented_exploratory_data_analysis.ipynb`, en este notebook se realizará:

- Carga del dataset raw de burnout.
- Eliminación de variables que no deben utilizarse como predictoras (`user_id`, `burnout_score`).
- División en conjunto de entrenamiento y de test **antes** de ajustar cualquier transformación, para evitar *data leakage*.
- Construcción de un `Pipeline` de scikit-learn con `ColumnTransformer` para codificar variables categóricas y escalar variables numéricas.
- Serialización del pipeline ajustado para que pueda reutilizarse en el resto del proyecto.
- Implementación de la función `clean_and_process(raw_dict)`, que recibirá el diccionario proveniente del formulario HTML (Miembro 5) y devolverá un array listo para el modelo (Miembro 3).

---

## EN | Goal

The goal of this notebook is to transform the raw burnout dataset into data that is ready for Machine Learning training.

Building on the findings from `01_business_oriented_exploratory_data_analysis.ipynb`, this notebook will:

- Load the raw burnout dataset.
- Remove variables that should not be used as predictors (`user_id`, `burnout_score`).
- Split the data into train and test sets **before** fitting any transformation, to avoid data leakage.
- Build a scikit-learn `Pipeline` with a `ColumnTransformer` to encode categorical variables and scale numerical variables.
- Serialize the fitted pipeline so it can be reused across the rest of the project.
- Implement the `clean_and_process(raw_dict)` function, which will receive the dictionary coming from the HTML form (Member 5) and return an array ready for the model (Member 3).


## 0. Google Colab Setup

### ES

Si este notebook se ejecuta en Google Colab, monta tu Google Drive o clona el repositorio del proyecto para que las rutas relativas (`../data/...`, `../models/...`) funcionen correctamente.

Si se ejecuta en local (por ejemplo dentro de la carpeta `notebooks/` del repositorio), esta celda puede omitirse.

### EN

If this notebook runs on Google Colab, mount your Google Drive or clone the project repository so that the relative paths (`../data/...`, `../models/...`) work correctly.

If it runs locally (e.g. inside the repository's `notebooks/` folder), this cell can be skipped.


In [6]:
# ES: Descomenta y ejecuta esta celda solo si trabajas en Google Colab.
# EN: Uncomment and run this cell only if you are working on Google Colab.

# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/SyntaxRelax/notebooks

# ES: Alternativa - clonar el repositorio del proyecto directamente en Colab.
# EN: Alternative - clone the project repository directly in Colab.

# !git clone https://github.com/tu-usuario/tu-repositorio.git
# %cd tu-repositorio/notebooks


## 1. Setup

### ES

Importamos las librerías necesarias para el preprocesado: `pandas` y `numpy` para manipulación de datos, y los módulos de `scikit-learn` para construir el pipeline de transformación.

### EN

We import the libraries required for preprocessing: `pandas` and `numpy` for data manipulation, and the `scikit-learn` modules to build the transformation pipeline.


In [7]:
# ES: Librerías principales.
# EN: Main libraries.

import pandas as pd
import numpy as np
import joblib
import json
import os

# ES: Librerías de scikit-learn para el pipeline de preprocesado.
# EN: scikit-learn libraries for the preprocessing pipeline.

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

# ES: Semilla para reproducibilidad.
# EN: Seed for reproducibility.

RANDOM_STATE = 42


## 2. Load Raw Dataset

### ES

Cargamos el dataset de burnout, que será la base para el modelo de estimación de riesgo de burnout.

Recordamos que, según el notebook 01, este dataset no presenta valores nulos ni duplicados, por lo que no será necesario aplicar limpieza adicional en ese sentido.

### EN

We load the burnout dataset, which will be the foundation for the burnout risk estimation model.

As shown in notebook 01, this dataset has no missing values or duplicated rows, so no additional cleaning is required in that respect.


In [8]:
# ES: Definir la ruta del dataset raw.
# EN: Define the raw dataset path.

burnout_path = "../data/01_output/burnout_df.csv"

# ES: Cargar el dataset.
# EN: Load the dataset.

burnout_df = pd.read_csv(burnout_path)

print(f"Shape: {burnout_df.shape}")
display(burnout_df.head())


FileNotFoundError: [Errno 2] No such file or directory: '../data/01_output/burnout_df.csv'

## 3. Recap of EDA Decisions

### ES

A partir de las conclusiones del notebook 01, aplicamos las siguientes decisiones antes de construir el pipeline:

- `user_id` se elimina: es un identificador de empleado, no aporta información predictiva.
- `burnout_score` se elimina: `burnout_risk` se deriva directamente de esta variable, por lo que utilizarla como predictor introduciría **data leakage**.
- `burnout_risk` es la variable objetivo (*target*), con tres categorías ordinales: `Low`, `Medium`, `High`.
- El resto de variables (`day_type` como categórica, y las variables de comportamiento laboral como numéricas) se utilizarán como predictoras.

### EN

Based on the conclusions from notebook 01, we apply the following decisions before building the pipeline:

- `user_id` is removed: it is an employee identifier and does not provide predictive information.
- `burnout_score` is removed: `burnout_risk` is directly derived from this variable, so using it as a predictor would introduce **data leakage**.
- `burnout_risk` is the target variable, with three ordinal categories: `Low`, `Medium`, `High`.
- The remaining variables (`day_type` as categorical, and the work-behaviour variables as numerical) will be used as predictors.


In [ ]:
# ES: Variable objetivo.
# EN: Target variable.

target_column = "burnout_risk"

# ES: Columnas a excluir del conjunto de predictores.
# EN: Columns excluded from the predictor set.

columns_to_drop = ["user_id", "burnout_score", target_column]

# ES: Variable categórica.
# EN: Categorical feature.

categorical_features = ["day_type"]

# ES: Variables numéricas (todo lo que no sea categórico, target o excluido).
# EN: Numerical features (everything that is not categorical, target or excluded).

numerical_features = [
    column for column in burnout_df.columns
    if column not in columns_to_drop + categorical_features
]

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)


Categorical features: ['day_type']
Numerical features: ['work_hours', 'screen_time_hours', 'meetings_count', 'breaks_taken', 'after_hours_work', 'app_switches', 'sleep_hours', 'task_completion', 'isolation_index', 'fatigue_score']


## 4. Train / Test Split

### ES

Dividimos el dataset en conjunto de entrenamiento y de test **antes** de ajustar cualquier transformación (escalado, codificación).

Esto es fundamental para evitar *data leakage*: si ajustásemos el `StandardScaler` o el `OneHotEncoder` con todo el dataset, el conjunto de test dejaría de ser una evaluación realmente independiente.

Utilizamos `stratify=y` para conservar la proporción de las clases `Low`, `Medium` y `High` en ambos conjuntos, dado el desbalanceo detectado en el notebook 01 (la clase `High` representa solo ~7% de los datos).

### EN

We split the dataset into train and test sets **before** fitting any transformation (scaling, encoding).

This is essential to avoid data leakage: if we fit the `StandardScaler` or `OneHotEncoder` using the whole dataset, the test set would no longer be a truly independent evaluation.

We use `stratify=y` to preserve the proportion of the `Low`, `Medium` and `High` classes in both sets, given the class imbalance identified in notebook 01 (the `High` class represents only ~7% of the data).


In [ ]:
# ES: Separar predictores (X) y variable objetivo (y).
# EN: Separate predictors (X) and target variable (y).

X = burnout_df[categorical_features + numerical_features]
y = burnout_df[target_column]

# ES: División estratificada en train (80%) y test (20%).
# EN: Stratified split into train (80%) and test (20%).

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print("\nClass distribution (train):")
print(y_train.value_counts(normalize=True).round(3))
print("\nClass distribution (test):")
print(y_test.value_counts(normalize=True).round(3))


X_train: (1600, 11)  |  X_test: (400, 11)

Class distribution (train):
burnout_risk
Low       0.509
Medium    0.421
High      0.069
Name: proportion, dtype: float64

Class distribution (test):
burnout_risk
Low       0.510
Medium    0.422
High      0.068
Name: proportion, dtype: float64


## 5. Preprocessing Pipeline

### ES

Construimos el `ColumnTransformer` que aplicará:

- `OneHotEncoder` sobre la variable categórica `day_type`, con `handle_unknown="ignore"` para evitar errores si en producción llega una categoría no vista durante el entrenamiento.
- `StandardScaler` sobre las variables numéricas, para que todas tengan media 0 y desviación estándar 1. Esto es especialmente relevante para modelos sensibles a la escala como la Regresión Logística, que se comparará en el notebook 03.

Todo el proceso se encapsula en un objeto `Pipeline` de scikit-learn para poder reutilizarlo de forma consistente durante el entrenamiento, la evaluación y la aplicación final.

### EN

We build the `ColumnTransformer` that will apply:

- `OneHotEncoder` on the categorical variable `day_type`, with `handle_unknown="ignore"` to avoid errors if an unseen category appears in production.
- `StandardScaler` on the numerical variables, so that all of them have mean 0 and standard deviation 1. This is especially relevant for scale-sensitive models such as Logistic Regression, which will be compared in notebook 03.

The whole process is wrapped in a scikit-learn `Pipeline` object so it can be reused consistently across training, evaluation and the final application.


In [ ]:
# ES: Definir las transformaciones por tipo de variable.
# EN: Define the transformations per feature type.

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), categorical_features),
        ("numerical", StandardScaler(), numerical_features),
    ]
)

# ES: Encapsular el preprocesado en un Pipeline.
# EN: Wrap the preprocessing step in a Pipeline.

preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

preprocessing_pipeline


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore'),
                                                  ['day_type']),
                                                 ('numerical', StandardScaler(),
                                                  ['work_hours',
                                                   'screen_time_hours',
                                                   'meetings_count',
                                                   'breaks_taken',
                                                   'after_hours_work',
                                                   'app_switches',
                                                   'sleep_hours',
                                                   'task_completion',
                                                   'isolation_index',
                                                   'fatigue_score'])]))])

In [ ]:
# ES: Ajustar el pipeline únicamente con datos de entrenamiento.
# EN: Fit the pipeline using only training data.

preprocessing_pipeline.fit(X_train)

# ES: Transformar train y test con el pipeline ya ajustado.
# EN: Transform train and test with the already-fitted pipeline.

X_train_processed = preprocessing_pipeline.transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

# ES: Obtener los nombres de las columnas resultantes tras la transformación.
# EN: Get the resulting column names after transformation.

feature_names = preprocessing_pipeline.named_steps["preprocessor"].get_feature_names_out()

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")
print("\nResulting features:")
print(list(feature_names))


X_train_processed shape: (1600, 11)
X_test_processed shape: (400, 11)

Resulting features:
['categorical__day_type_Weekend', 'numerical__work_hours', 'numerical__screen_time_hours', 'numerical__meetings_count', 'numerical__breaks_taken', 'numerical__after_hours_work', 'numerical__app_switches', 'numerical__sleep_hours', 'numerical__task_completion', 'numerical__isolation_index', 'numerical__fatigue_score']


### Interpretation

#### ES

Tras aplicar el pipeline, la variable `day_type` se ha transformado en una columna binaria (codificación *one-hot*), y las variables numéricas se han escalado para tener media 0 y desviación estándar 1.

El número de columnas resultante es ligeramente superior al original, debido a la codificación de `day_type`. Este conjunto de datos ya está listo para ser utilizado por cualquier algoritmo de Machine Learning sensible a la escala de las variables.

#### EN

After applying the pipeline, the `day_type` variable has been transformed into a binary column (one-hot encoding), and the numerical variables have been scaled to have mean 0 and standard deviation 1.

The resulting number of columns is slightly higher than the original, due to the encoding of `day_type`. This dataset is now ready to be used by any Machine Learning algorithm that is sensitive to feature scale.


## 6. Save Processed Data and Fitted Pipeline

### ES

Guardamos:

- El pipeline de preprocesado ya ajustado (`preprocessing_pipeline.joblib`), para que Miembro 3 (ML Engineer) pueda transformar nuevos datos de forma consistente.
- Los conjuntos de entrenamiento y test ya procesados, para que puedan reutilizarse directamente en el notebook 03 sin tener que repetir el preprocesado.

### EN

We save:

- The already-fitted preprocessing pipeline (`preprocessing_pipeline.joblib`), so Member 3 (ML Engineer) can transform new data consistently.
- The already-processed train and test sets, so they can be reused directly in notebook 03 without repeating the preprocessing step.


In [ ]:
# ES: Crear las carpetas de salida si no existen.
# EN: Create the output folders if they do not exist.

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# ES: Guardar el pipeline de preprocesado ajustado.
# EN: Save the fitted preprocessing pipeline.

joblib.dump(preprocessing_pipeline, "../models/preprocessing_pipeline.joblib")

# ES: Guardar los datos ya procesados en formato numpy comprimido.
# EN: Save the already-processed data in compressed numpy format.

np.savez(
    "../data/processed/burnout_processed.npz",
    X_train=X_train_processed,
    X_test=X_test_processed,
    y_train=y_train.values,
    y_test=y_test.values,
)

# ES: Guardar los nombres de las columnas y el orden de features esperado por el pipeline.
# EN: Save the column names and the feature order expected by the pipeline.

metadata = {
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "feature_names_out": list(feature_names),
    "target_classes": ["Low", "Medium", "High"],
}

with open("../models/preprocessing_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Pipeline, processed data and metadata saved successfully.")


Pipeline, processed data and metadata saved successfully.


## 7. `clean_and_process(raw_dict)` Function

### ES

Esta es la entrega principal de Miembro 2 (Data Engineer) para el resto del equipo.

La función `clean_and_process(raw_dict)`:

1. Recibe un diccionario con las claves exactas de las variables de entrada (las mismas que usará el formulario HTML de Miembro 5).
2. Construye un DataFrame de una sola fila, respetando el orden de columnas esperado por el pipeline.
3. Aplica el pipeline de preprocesado ya ajustado (codificación + escalado).
4. Devuelve un array de NumPy listo para ser pasado a `predict_attrition()` (Miembro 3).

Esta misma función se guardará en `src/preprocessing.py` para que pueda importarse desde `app.py` (Miembro 5) sin depender de este notebook.

### EN

This is Member 2's (Data Engineer) main deliverable for the rest of the team.

The `clean_and_process(raw_dict)` function:

1. Receives a dictionary with the exact keys of the input variables (the same ones used by Member 5's HTML form).
2. Builds a single-row DataFrame, respecting the column order expected by the pipeline.
3. Applies the already-fitted preprocessing pipeline (encoding + scaling).
4. Returns a NumPy array ready to be passed to `predict_attrition()` (Member 3).

This same function will be saved in `src/preprocessing.py` so it can be imported from `app.py` (Member 5) without depending on this notebook.


In [ ]:
# ES: Cargar el pipeline guardado (simula cómo lo haría app.py en producción).
# EN: Load the saved pipeline (simulates how app.py would do it in production).

_fitted_pipeline = joblib.load("../models/preprocessing_pipeline.joblib")

with open("../models/preprocessing_metadata.json") as f:
    _metadata = json.load(f)

_expected_columns = _metadata["categorical_features"] + _metadata["numerical_features"]


def clean_and_process(raw_dict, pipeline=_fitted_pipeline, expected_columns=_expected_columns):
    """
    ES: Transforma un diccionario de entrada (por ejemplo, procedente de un
    formulario HTML) en un array listo para el modelo de riesgo de burnout.

    EN: Transforms an input dictionary (e.g. coming from an HTML form)
    into an array ready for the burnout risk model.

    Parameters
    ----------
    raw_dict : dict
        ES: Diccionario con las claves: day_type, work_hours, screen_time_hours,
        meetings_count, breaks_taken, after_hours_work, app_switches,
        sleep_hours, task_completion, isolation_index, fatigue_score.
        EN: Dictionary with keys: day_type, work_hours, screen_time_hours,
        meetings_count, breaks_taken, after_hours_work, app_switches,
        sleep_hours, task_completion, isolation_index, fatigue_score.

    Returns
    -------
    numpy.ndarray
        ES: Array 2D (1 fila) listo para pasar a predict_attrition().
        EN: 2D array (1 row) ready to be passed to predict_attrition().
    """

    # ES: Comprobar que no falte ninguna clave esperada.
    # EN: Check that no expected key is missing.

    missing_keys = [column for column in expected_columns if column not in raw_dict]
    if missing_keys:
        raise ValueError(f"Missing keys in raw_dict: {missing_keys}")

    # ES: Construir un DataFrame de una fila con el orden de columnas correcto.
    # EN: Build a single-row DataFrame with the correct column order.

    input_df = pd.DataFrame([{column: raw_dict[column] for column in expected_columns}])

    # ES: Aplicar el pipeline de preprocesado ya ajustado.
    # EN: Apply the already-fitted preprocessing pipeline.

    processed_array = pipeline.transform(input_df)

    return processed_array


### Testing the function

#### ES

Probamos la función con un ejemplo simulado, similar al que llegaría desde el formulario web: un puesto con jornada larga, poco descanso y alta fatiga (perfil de riesgo esperado: medio/alto).

#### EN

We test the function with a simulated example, similar to what would arrive from the web form: a job profile with long working hours, little rest and high fatigue (expected risk profile: medium/high).


In [ ]:
# ES: Ejemplo de diccionario de entrada, como el que enviaría el formulario HTML.
# EN: Example input dictionary, similar to what the HTML form would send.

sample_raw_dict = {
    "day_type": "Weekday",
    "work_hours": 11.5,
    "screen_time_hours": 10.2,
    "meetings_count": 5,
    "breaks_taken": 2,
    "after_hours_work": 1,
    "app_switches": 95,
    "sleep_hours": 5.2,
    "task_completion": 68.0,
    "isolation_index": 7,
    "fatigue_score": 8.6,
}

processed_sample = clean_and_process(sample_raw_dict)

print("Processed array shape:", processed_sample.shape)
print(processed_sample)


Processed array shape: (1, 11)
[[ 0.          1.47111     1.25694751  0.45097387 -1.71333312  2.34607439
   1.62773544 -1.72901068 -0.93982067  1.27037338  0.92324613]]


### Business Insight

#### ES

La función `clean_and_process` desacopla la lógica de transformación de datos del resto de la aplicación. Esto tiene varias ventajas para el proyecto:

- Miembro 5 (Flask) puede llamar a esta función sin necesitar conocer los detalles internos del `ColumnTransformer` o del escalado.
- Miembro 3 (ML Engineer) recibe siempre un array con la misma estructura, independientemente de cómo evolucione el preprocesado.
- Si en el futuro se añaden nuevas variables o se cambia el tipo de codificación, solo será necesario modificar esta función y volver a guardar el pipeline, sin tocar el resto del código de la aplicación.

#### EN

The `clean_and_process` function decouples the data transformation logic from the rest of the application. This has several advantages for the project:

- Member 5 (Flask) can call this function without needing to know the internal details of the `ColumnTransformer` or the scaling.
- Member 3 (ML Engineer) always receives an array with the same structure, regardless of how the preprocessing evolves.
- If new variables are added in the future, or the encoding strategy changes, only this function needs to be modified and the pipeline re-saved, without touching the rest of the application code.


## 8. Export `clean_and_process` to `src/preprocessing.py`

### ES

Por último, guardamos el código fuente de la función en `src/preprocessing.py`, tal y como se especifica en el reparto de tareas del equipo. Este archivo podrá importarse directamente desde `app.py` (`from preprocessing import clean_and_process`).

### EN

Finally, we save the function's source code in `src/preprocessing.py`, as specified in the team's task breakdown. This file can be imported directly from `app.py` (`from preprocessing import clean_and_process`).


In [9]:
preprocessing_py_content = '''"""
ES: Módulo de preprocesado de datos para la app de riesgo de burnout.
EN: Data preprocessing module for the burnout risk app.

Miembro 2 - Anas: Data Engineer
"""

import json
import joblib
import pandas as pd

_PIPELINE_PATH = "models/preprocessing_pipeline.joblib"
_METADATA_PATH = "models/preprocessing_metadata.json"

_fitted_pipeline = joblib.load(_PIPELINE_PATH)

with open(_METADATA_PATH) as f:
    _metadata = json.load(f)

_expected_columns = _metadata["categorical_features"] + _metadata["numerical_features"]


def clean_and_process(raw_dict, pipeline=_fitted_pipeline, expected_columns=_expected_columns):
    """
    ES: Transforma un diccionario de entrada (formulario HTML) en un array
    listo para el modelo de riesgo de burnout.
    EN: Transforms an input dictionary (HTML form) into an array ready
    for the burnout risk model.
    """

    missing_keys = [column for column in expected_columns if column not in raw_dict]
    if missing_keys:
        raise ValueError(f"Missing keys in raw_dict: {missing_keys}")

    input_df = pd.DataFrame([{column: raw_dict[column] for column in expected_columns}])

    return pipeline.transform(input_df)
'''

os.makedirs("../src", exist_ok=True)

with open("../src/preprocessing.py", "w") as f:
    f.write(preprocessing_py_content)

print("src/preprocessing.py written successfully.")


src/preprocessing.py written successfully.


## 9. Conclusions and Handoff to Member 3

### ES

En este notebook se ha construido y validado el pipeline de preprocesado del dataset de burnout:

- Se han excluido `user_id` (identificador) y `burnout_score` (fuente de *data leakage*).
- Se ha dividido el dataset en train/test de forma estratificada, antes de ajustar cualquier transformación.
- Se ha construido un `ColumnTransformer` con `OneHotEncoder` para `day_type` y `StandardScaler` para las variables numéricas.
- Se han guardado el pipeline ajustado, los datos procesados y la función `clean_and_process`, lista para ser reutilizada por el resto del equipo.

**Entrega para Miembro 3 (ML Engineer):** los ficheros `../models/preprocessing_pipeline.joblib` y `../data/processed/burnout_processed.npz` contienen todo lo necesario para comenzar el entrenamiento de modelos en el notebook `03_model_selection_and_training.ipynb`.

### EN

This notebook has built and validated the preprocessing pipeline for the burnout dataset:

- `user_id` (identifier) and `burnout_score` (source of data leakage) have been excluded.
- The dataset has been split into train/test in a stratified way, before fitting any transformation.
- A `ColumnTransformer` has been built with `OneHotEncoder` for `day_type` and `StandardScaler` for the numerical variables.
- The fitted pipeline, the processed data and the `clean_and_process` function have been saved, ready to be reused by the rest of the team.

**Handoff to Member 3 (ML Engineer):** the files `../models/preprocessing_pipeline.joblib` and `../data/processed/burnout_processed.npz` contain everything needed to start model training in `03_model_selection_and_training.ipynb`.
